In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import roc_auc_score
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.preprocessing import StandardScaler

### Конфигурация и Воспроизводимость
Фиксируем `random_seed` для всех библиотек, чтобы эксперимент был полностью воспроизводим. Все гиперпараметры вынесены в класс `Config`.
- `batch_size=64`: Баланс между скоростью обучения и стабильностью градиента.
- `epochs=100`: Достаточное количество эпох для работы циклического шедулера (несколько полных циклов).


In [2]:
class Config:
    seed = 42
    batch_size = 64
    epochs = 100
    lr = 1e-2
    input_dim = None

config = Config()
pl.seed_everything(config.seed)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

### Компоненты модели
Реализация **Residual Block** для табличных данных.
Структура: `Linear -> BN -> ReLU -> Dropout -> Linear -> BN -> ReLU -> Dropout`.
Skip-connection (`x + block(x)`) позволяет сигналу проходить сквозь слои беспрепятственно. `Dropout` (0.1) используется для регуляризации и предотвращения переобучения на небольшом датасете.


In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
    def forward(self, x): return x + self.block(x)

### Основной класс модели

- **Loss:** Используется `BCEWithLogitsLoss`, так как он численно более стабилен, чем Sigmoid + BCELoss.
- **Метрика:** Валидация проводится по **ROC AUC**.
- **Оптимизатор:** `AdamW` с `CosineAnnealingWarmRestarts` для эффективной сходимости.


In [4]:
class TabularResNet(pl.LightningModule):
    def __init__(self, input_dim, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(),
            ResidualBlock(512),
            ResidualBlock(512),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
        self.criterion = nn.BCEWithLogitsLoss()
        self.validation_step_outputs = []

    def forward(self, x): return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        probs = torch.sigmoid(self(x))
        self.validation_step_outputs.append((probs, y))

    def on_validation_epoch_end(self):
        if not self.validation_step_outputs: return
        probs = torch.cat([x[0] for x in self.validation_step_outputs]).cpu()
        targets = torch.cat([x[1] for x in self.validation_step_outputs]).cpu()
        try:
            auc = roc_auc_score(targets, probs)
            self.log('val_auc', auc, prog_bar=True)
        except: pass
        self.validation_step_outputs.clear()

    def predict_step(self, batch, batch_idx):
        return torch.sigmoid(self(batch))

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.hparams.lr)
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_auc"}}


### Подготовка данных

In [5]:
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')



### Предобработка и Feature Engineering
1. **Очистка:** Удаление аномалий
2. **Скалирование:** Применение `StandardScaler` для приведения всех признаков к нормальному распределению.


In [6]:

for col in ['eyesight(left)', 'eyesight(right)']:
    mask = train_df[col] > 9.0
    median = train_df.loc[~mask, col].median()
    train_df.loc[mask, col] = median
    test_df.loc[test_df[col] > 9.0, col] = median

features = [c for c in train_df.columns if c not in ['id', 'smoking']]
config.input_dim = len(features)

X = train_df[features].values
y = train_df['smoking'].values
X_test = test_df[features].values

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)


full_ds = TabularDataset(X, y)
train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size], generator=torch.Generator().manual_seed(42))
test_ds = TabularDataset(X_test)

train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, num_workers=2)

### Обучение модели
Инициализация `Trainer` с колбэками:
- **ModelCheckpoint:** Сохраняет веса модели с лучшим показателем `val_auc`.
- **EarlyStopping:** Останавливает обучение, если метрика не растет в течение 10 эпох.


In [7]:
model = TabularResNet(input_dim=config.input_dim, lr=config.lr)
checkpoint_callback = ModelCheckpoint(monitor='val_auc', mode='max', save_top_k=1, filename='best_single_model', verbose=True)
early_stop = EarlyStopping(monitor='val_auc', patience=10, mode='max')

trainer = pl.Trainer(max_epochs=config.epochs, accelerator="auto", devices=1, callbacks=[checkpoint_callback, early_stop])
trainer.fit(model, train_loader, val_loader)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net       │ Sequential        │  1.2 M │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.2 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.2 M                                                                                                
Total estimated model params size (MB): 4                                                                          
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:Epoch 0, global step 188: 'val_auc' reached 0.87858 (best 0.87858), saving model to '/content/lightning_logs/version_0/checkpoints/best_single_model.ckpt' as top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 1, global step 376: 'val_auc' reached 0.87946 (best 0.87946), saving model to '/content/lightning_logs/version_0/checkpoints/best_single_model.ckpt' as top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 2, global step 564: 'val_auc' reached 0.88081 (best 0.88081), saving model to '/content/lightning_logs/version_0/checkpoints/best_single_model.ckpt' as top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 3, global step 752: 'val_auc' reached 0.88254 (best 0.88254), saving model to '/content/lightning_logs/version_0/checkpoints/best_single_model.ckpt' as top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 4, global step 940: 'val_auc' was not in top 1
INFO:pytorch_lightning.utilities.rank_zero:Epoch 5, global step 1128: 'val_a

### Генерация сабмита
1. Загрузка весов лучшей модели (`best_model_path`).
2. Предсказание вероятностей на тестовом наборе.
3. Формирование файла `submission.csv` в формате `id,smoking`.


In [8]:

best_model = TabularResNet.load_from_checkpoint(checkpoint_callback.best_model_path)
preds = trainer.predict(best_model, dataloaders=test_loader)
preds_flat = torch.cat(preds).cpu().numpy().flatten()

submission = pd.DataFrame({'id': test_df['id'], 'smoking': preds_flat})
submission.to_csv('submission.csv', index=False)



INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()